# KOA Classifier — VGG19 + CLAHE + GA + RF + SVM 200 Fitur

**Skenario:** Balancing Training Set Berdasarkan Kelas Minoritas

## 1 — Setup Dependency, Path, dan Konfigurasi

In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import re
import time
from collections import Counter
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple
from types import SimpleNamespace

import cv2
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
    cohen_kappa_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, StratifiedShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from types import SimpleNamespace
from pathlib import Path

DATA_ROOT = r"C:/model_skripsi/archive"
OUT_DIR = r"C:/model_skripsi/outputs_minor_augmented"

args = SimpleNamespace(
    data_root=DATA_ROOT,
    out_dir=OUT_DIR,

    seed=100,
    train_size=0.75,
    val_size=0.1,
    test_size=0.15,

    batch_size=24,
    epochs_head=30,
    epochs_finetune=12,
    early_stop_patience=8,

    lr_head=1e-4,
    lr_finetune=1e-5,
    dropout=0.55,
    l2=1e-4,
    bottleneck_dim=200,

    ga_population=100,
    ga_generations=50,
    ga_min_features=10,
    ga_init_prob=0.45,
)

print("DATA_ROOT:", args.data_root)
print("OUT_DIR:", args.out_dir)

CLASS_NAMES = {
    0: "Grade 0 - Normal",
    1: "Grade 1 - Doubtful",
    2: "Grade 2 - Mild",
    3: "Grade 3 - Moderate",
    4: "Grade 4 - Severe",
}

CLASS_ALIASES = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "grade0": 0,
    "grade1": 1,
    "grade2": 2,
    "grade3": 3,
    "grade4": 4,
    "grade_0": 0,
    "grade_1": 1,
    "grade_2": 2,
    "grade_3": 3,
    "grade_4": 4,
    "normal": 0,
    "doubtful": 1,
    "mild": 2,
    "moderate": 3,
    "severe": 4,
}

IMG_SIZE = (224, 224)
NUM_CLASSES = 5
CLAHE_CLIP_LIMIT = 3.0
CLAHE_TILE_GRID_SIZE = (8, 8)

AUGMENTATION_ENABLED = True
AUGMENTATION_ROTATION_DEGREE = 10.0
AUGMENTATION_ZOOM_RANGE = 0.20
AUGMENTATION_HORIZONTAL_FLIP = True
AUGMENTATION_FILL_MODE = "nearest"

def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

## 2 — Dataset Scanning, Split, dan Balancing Training Set

Cell ini berisi fungsi pembacaan dataset, pembuatan `group_id`, stratified split, dan balancing khusus data training sesuai skenario notebook.

In [ ]:
def normalize_label_name(name: str) -> str:
    name = name.strip().lower()
    name = name.replace(" ", "")
    name = name.replace("-", "_")
    return name

def infer_label_from_path(path: Path, data_root: Path) -> Optional[int]:
    try:
        rel_parts = path.relative_to(data_root).parts
    except ValueError:
        rel_parts = path.parts

    for part in reversed(rel_parts[:-1]):
        key = normalize_label_name(part)
        if key in CLASS_ALIASES:
            return CLASS_ALIASES[key]

    return None

def infer_group_id(path: Path) -> str:
    stem = path.stem

    s = re.sub(r"([_-]?(left|right|LEFT|RIGHT))$", "", stem)
    s = re.sub(r"([_-]?[LRlr])$", "", s)

    numeric = re.findall(r"\d{5,}", s)
    if numeric:
        return numeric[0]

    return s

def scan_dataset(data_root: Path) -> pd.DataFrame:
    image_exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    rows = []

    for path in data_root.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in image_exts:
            continue

        label = infer_label_from_path(path, data_root)
        if label is None:
            continue

        rows.append(
            {
                "filepath": str(path),
                "label": int(label),
                "group_id": infer_group_id(path),
                "filename": path.name,
            }
        )

    df = pd.DataFrame(rows)

    if df.empty:
        raise RuntimeError(
            f"Tidak menemukan gambar berlabel 0-4 di: {data_root}. "
            "Pastikan data_root mengarah ke folder dataset yang benar."
        )

    df = df.drop_duplicates(subset=["filepath"]).reset_index(drop=True)
    return df

def group_label_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for gid, g in df.groupby("group_id"):
        counts = g["label"].value_counts()
        rows.append(
            {
                "group_id": gid,
                "group_label": int(counts.index[0]),
                "n_images": int(len(g)),
                "labels_in_group": ",".join(map(str, sorted(g["label"].unique().tolist()))),
            }
        )
    return pd.DataFrame(rows)

def safe_stratified_group_split(
    df: pd.DataFrame,
    train_size: float,
    val_size: float,
    test_size: float,
    seed: int,
    min_groups_per_class: int = 3,
    force_image_level: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    assert abs(train_size + val_size + test_size - 1.0) < 1e-6

    def image_level_split(reason: str):
        print("[WARNING] Group-aware split tidak digunakan.")
        print("[WARNING] Alasan:", reason)
        print("[WARNING] Fallback ke image-level stratified split.")
        print("[WARNING] Catatan: gunakan hasil split_leakage_audit untuk mengecek potensi overlap group_id.")

        train_df, temp_df = train_test_split(
            df,
            train_size=train_size,
            stratify=df["label"],
            random_state=seed,
        )

        relative_val = val_size / (val_size + test_size)

        val_df, test_df = train_test_split(
            temp_df,
            train_size=relative_val,
            stratify=temp_df["label"],
            random_state=seed,
        )

        return (
            train_df.sample(frac=1, random_state=seed).reset_index(drop=True),
            val_df.sample(frac=1, random_state=seed).reset_index(drop=True),
            test_df.sample(frac=1, random_state=seed).reset_index(drop=True),
        )

    if force_image_level:
        return image_level_split("force_image_level=True")

    gtab = group_label_table(df)

    print("Distribusi label pada level image:")
    print(df["label"].value_counts().sort_index())

    print("Distribusi label pada level group_id:")
    print(gtab["group_label"].value_counts().sort_index())

    group_counts = gtab["group_label"].value_counts().sort_index()
    if gtab["group_label"].nunique() < NUM_CLASSES:
        return image_level_split(
            "Tidak semua kelas 0-4 muncul pada level group_id."
        )

    if group_counts.min() < min_groups_per_class:
        return image_level_split(
            f"Jumlah group minimum per kelas hanya {int(group_counts.min())}, "
            f"kurang dari batas aman {min_groups_per_class}."
        )

    try:
        train_groups, temp_groups = train_test_split(
            gtab,
            train_size=train_size,
            stratify=gtab["group_label"],
            random_state=seed,
        )

        relative_val = val_size / (val_size + test_size)

        temp_counts = temp_groups["group_label"].value_counts()
        if temp_counts.min() < 2:
            return image_level_split(
                "Setelah train/temp split, ada kelas pada temp yang hanya memiliki 1 group, "
                "sehingga val/test stratified split tidak memungkinkan."
            )

        val_groups, test_groups = train_test_split(
            temp_groups,
            train_size=relative_val,
            stratify=temp_groups["group_label"],
            random_state=seed,
        )

        train_df = df[df["group_id"].isin(train_groups["group_id"])].copy()
        val_df = df[df["group_id"].isin(val_groups["group_id"])].copy()
        test_df = df[df["group_id"].isin(test_groups["group_id"])].copy()

        assert set(train_df["group_id"]).isdisjoint(set(val_df["group_id"]))
        assert set(train_df["group_id"]).isdisjoint(set(test_df["group_id"]))
        assert set(val_df["group_id"]).isdisjoint(set(test_df["group_id"]))

        return (
            train_df.sample(frac=1, random_state=seed).reset_index(drop=True),
            val_df.sample(frac=1, random_state=seed).reset_index(drop=True),
            test_df.sample(frac=1, random_state=seed).reset_index(drop=True),
        )

    except ValueError as e:
        return image_level_split(str(e))

def print_distribution(name: str, df: pd.DataFrame) -> None:
    counts = df["label"].value_counts().sort_index()
    print(f"\n{name}: n={len(df)}, groups={df['group_id'].nunique()}")
    for cls in range(NUM_CLASSES):
        print(f"  Grade {cls}: {int(counts.get(cls, 0))}")

def compute_sample_weight_map(y: np.ndarray) -> Dict[int, float]:
    counts = Counter(y.tolist())
    total = len(y)
    n_classes = len(counts)
    weights = {}
    for cls, count in counts.items():
        weights[int(cls)] = total / (n_classes * count)
    return weights

from sklearn.utils import resample

def balance_training_dataframe(
    train_df: pd.DataFrame,
    strategy: str = "minority",
    target_count: int = None,
    seed: int = 42,
) -> pd.DataFrame:
    class_counts = train_df["label"].value_counts().sort_index()

    if strategy == "minority":
        target = int(class_counts.min())

    elif strategy == "majority":
        target = int(class_counts.max())

    elif strategy == "fixed":
        if target_count is None:
            raise ValueError("target_count wajib diisi untuk strategy='fixed'")
        target = int(target_count)

    else:
        raise ValueError(f"Strategy tidak dikenal: {strategy}")

    balanced_parts = []

    for cls in sorted(train_df["label"].unique()):
        cls_df = train_df[train_df["label"] == cls]

        if len(cls_df) > target:
            sampled_df = resample(
                cls_df,
                replace=False,
                n_samples=target,
                random_state=seed,
            )

        elif len(cls_df) < target:
            sampled_df = resample(
                cls_df,
                replace=True,
                n_samples=target,
                random_state=seed,
            )

        else:
            sampled_df = cls_df.copy()

        balanced_parts.append(sampled_df)

    balanced_df = pd.concat(balanced_parts, ignore_index=True)

    balanced_df = balanced_df.sample(
        frac=1.0,
        random_state=seed,
    ).reset_index(drop=True)

    print("\nDistribusi training set setelah balancing:")
    print(balanced_df["label"].value_counts().sort_index())

    return balanced_df

## 3 — Preprocessing Citra dan Data Generator

Preprocessing dengan memakai crop ROI, resize 224×224, CLAHE, dan augmentasi training.

In [ ]:
def resize_force_rgb(image_rgb: np.ndarray, target_size: Tuple[int, int] = IMG_SIZE) -> np.ndarray:
    return cv2.resize(
        image_rgb,
        target_size,
        interpolation=cv2.INTER_CUBIC
    )

def apply_clahe_rgb(image_rgb: np.ndarray) -> np.ndarray:
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=CLAHE_CLIP_LIMIT,
        tileGridSize=CLAHE_TILE_GRID_SIZE
    )
    l_clahe = clahe.apply(l_channel)

    lab_clahe = cv2.merge((l_clahe, a_channel, b_channel))
    image_bgr_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2BGR)
    image_rgb_clahe = cv2.cvtColor(image_bgr_clahe, cv2.COLOR_BGR2RGB)

    return image_rgb_clahe

def random_gamma(image_rgb: np.ndarray, gamma_range: Tuple[float, float] = (0.85, 1.15)) -> np.ndarray:
    gamma = np.random.uniform(*gamma_range)
    inv_gamma = 1.0 / gamma
    table = ((np.arange(256) / 255.0) ** inv_gamma * 255).astype(np.uint8)
    return cv2.LUT(image_rgb, table)

def random_brightness_contrast(image_rgb: np.ndarray) -> np.ndarray:
    alpha = np.random.uniform(0.90, 1.10)  # contrast
    beta = np.random.uniform(-10, 10)      # brightness
    return cv2.convertScaleAbs(image_rgb, alpha=alpha, beta=beta)

def random_affine(image_rgb: np.ndarray) -> np.ndarray:
    h, w = image_rgb.shape[:2]

    angle = np.random.uniform(
        -AUGMENTATION_ROTATION_DEGREE,
        AUGMENTATION_ROTATION_DEGREE,
    )
    scale = np.random.uniform(
        1.0 - AUGMENTATION_ZOOM_RANGE,
        1.0 + AUGMENTATION_ZOOM_RANGE,
    )

    center = (w / 2, h / 2)
    matrix = cv2.getRotationMatrix2D(center, angle, scale)

    return cv2.warpAffine(
        image_rgb,
        matrix,
        (w, h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE,
    )

def maybe_add_noise(image_rgb: np.ndarray, sigma_range: Tuple[float, float] = (0.0, 4.0)) -> np.ndarray:
    sigma = np.random.uniform(*sigma_range)
    if sigma <= 0.1:
        return image_rgb

    noise = np.random.normal(0, sigma, image_rgb.shape).astype(np.float32)
    out = image_rgb.astype(np.float32) + noise
    return np.clip(out, 0, 255).astype(np.uint8)

def augment_image(image_rgb: np.ndarray) -> np.ndarray:
    if AUGMENTATION_HORIZONTAL_FLIP and np.random.rand() < 0.50:
        image_rgb = cv2.flip(image_rgb, 1)

    image_rgb = random_affine(image_rgb)

    return image_rgb

def load_preprocess_image(path: str, augment: bool = False) -> np.ndarray:
    image_bgr = cv2.imread(path, cv2.IMREAD_COLOR)

    if image_bgr is None:
        raise ValueError(f"Gagal membaca citra: {path}")

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_rgb = resize_force_rgb(image_rgb, IMG_SIZE)

    if augment:
        image_rgb = augment_image(image_rgb)

    image_rgb = apply_clahe_rgb(image_rgb)

    image_float = image_rgb.astype(np.float32)
    image_float = tf.keras.applications.vgg19.preprocess_input(image_float)

    return image_float

class XraySequence(keras.utils.Sequence):
    def __init__(
        self,
        df: pd.DataFrame,
        batch_size: int,
        augment: bool,
        shuffle: bool,
        sample_weight_map: Optional[Dict[int, float]] = None,
    ):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.sample_weight_map = sample_weight_map
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self) -> int:
        return int(math.ceil(len(self.df) / self.batch_size))

    def on_epoch_end(self) -> None:
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx: int):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_indices]

        x = np.zeros((len(batch_df), IMG_SIZE[1], IMG_SIZE[0], 3), dtype=np.float32)
        y_int = batch_df["label"].astype(int).values

        for i, path in enumerate(batch_df["filepath"].values):
            x[i] = load_preprocess_image(path, augment=self.augment)

        y = keras.utils.to_categorical(y_int, num_classes=NUM_CLASSES)

        if self.sample_weight_map is not None:
            sw = np.array([self.sample_weight_map[int(label)] for label in y_int], dtype=np.float32)
            return x, y, sw

        return x, y

## 4 — Arsitektur VGG19 dan Bottleneck 200 Fitur

Model VGG19 menghasilkan bottleneck feature berdimensi 200 sebelum classifier akhir.

In [ ]:
def build_vgg19_model(
    bottleneck_dim: int = 200,
    dropout_rate: float = 0.55,
    l2_value: float = 1e-4,
) -> keras.Model:
    inputs = keras.Input(shape=(224, 224, 3), name="input_image")

    base = keras.applications.VGG19(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs,
    )
    base.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.BatchNormalization(name="bn_gap")(x)
    x = layers.Dropout(dropout_rate, name="dropout_gap")(x)

    x = layers.Dense(
        bottleneck_dim,
        activation="linear",
        kernel_regularizer=regularizers.l2(l2_value),
        name="bottleneck_features",
    )(x)
    x = layers.BatchNormalization(name="bn_bottleneck")(x)
    x = layers.Dropout(dropout_rate, name="dropout_bottleneck")(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        kernel_regularizer=regularizers.l2(l2_value),
        name="classification_output",
    )(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="vgg19_regularized_koa")
    return model

def set_vgg19_trainable_block5_only(model: keras.Model) -> None:
    for layer in model.layers:
        layer.trainable = False

    for layer in model.layers:
        if layer.name.startswith("block5_"):
            layer.trainable = True

        if layer.name in {
            "bn_gap",
            "dropout_gap",
            "bottleneck_features",
            "bn_bottleneck",
            "dropout_bottleneck",
            "classification_output",
        }:
            layer.trainable = True

def compile_model(model: keras.Model, learning_rate: float) -> None:
    try:
        optimizer = keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=1e-5,
        )
    except Exception:
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    loss = keras.losses.CategoricalCrossentropy(label_smoothing=0.05)

    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=[
            keras.metrics.CategoricalAccuracy(name="accuracy"),
            keras.metrics.AUC(name="auc", multi_label=True, num_labels=NUM_CLASSES),
        ],
    )

def save_model_summary(model: keras.Model, out_path: Path) -> None:
    lines = []
    model.summary(print_fn=lambda x: lines.append(x))
    out_path.write_text("\n".join(lines), encoding="utf-8")

## 5 — Evaluasi, Visualisasi, dan Export Output

Cell ini menyimpan metrik, confusion matrix, plot distribusi, preview preprocessing, dan CSV fitur.

In [ ]:
def safe_multiclass_auc(y_true: np.ndarray, proba: np.ndarray) -> Tuple[Optional[float], Optional[float]]:
    try:
        macro_auc = roc_auc_score(y_true, proba, multi_class="ovr", average="macro")
    except Exception:
        macro_auc = None

    try:
        weighted_auc = roc_auc_score(y_true, proba, multi_class="ovr", average="weighted")
    except Exception:
        weighted_auc = None

    return macro_auc, weighted_auc

def safe_binary_kl_auc(y_true: np.ndarray, proba: np.ndarray) -> Optional[float]:
    try:
        y_bin = (y_true >= 2).astype(int)
        p_bin = proba[:, 2:].sum(axis=1)
        return roc_auc_score(y_bin, p_bin)
    except Exception:
        return None

def classification_metrics(y_true: np.ndarray, y_pred: np.ndarray, proba: Optional[np.ndarray] = None) -> Dict[str, Optional[float]]:
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "quadratic_weighted_kappa": float(cohen_kappa_score(y_true, y_pred, weights="quadratic")),
        "mae_grade": float(mean_absolute_error(y_true, y_pred)),
        "mse_grade": float(mean_squared_error(y_true, y_pred)),
        "macro_auc_ovr": None,
        "weighted_auc_ovr": None,
        "binary_auc_kl_ge_2": None,
    }

    if proba is not None:
        macro_auc, weighted_auc = safe_multiclass_auc(y_true, proba)
        metrics["macro_auc_ovr"] = None if macro_auc is None else float(macro_auc)
        metrics["weighted_auc_ovr"] = None if weighted_auc is None else float(weighted_auc)

        binary_auc = safe_binary_kl_auc(y_true, proba)
        metrics["binary_auc_kl_ge_2"] = None if binary_auc is None else float(binary_auc)

    return metrics

def save_evaluation_outputs(
    name: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    proba: Optional[np.ndarray],
    out_dir: Path,
) -> Dict[str, Optional[float]]:
    out_dir.mkdir(parents=True, exist_ok=True)

    metrics = classification_metrics(y_true, y_pred, proba)

    with open(out_dir / f"{name}_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=[CLASS_NAMES[i] for i in range(NUM_CLASSES)],
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(report).T.to_csv(out_dir / f"{name}_classification_report.csv")

    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    pd.DataFrame(
        cm,
        index=[f"actual_{i}" for i in range(NUM_CLASSES)],
        columns=[f"pred_{i}" for i in range(NUM_CLASSES)],
    ).to_csv(out_dir / f"{name}_confusion_matrix.csv")

    plot_confusion_matrix_png(
        cm=cm,
        name=name,
        out_dir=out_dir,
    )

    if proba is not None:
        proba_df = pd.DataFrame(proba, columns=[f"proba_grade_{i}" for i in range(NUM_CLASSES)])
        proba_df["y_true"] = y_true
        proba_df["y_pred"] = y_pred
        proba_df.to_csv(out_dir / f"{name}_probabilities.csv", index=False)

    return metrics

def evaluate_cnn(model: keras.Model, df: pd.DataFrame, batch_size: int, name: str, out_dir: Path) -> Dict[str, Optional[float]]:
    seq = XraySequence(df, batch_size=batch_size, augment=False, shuffle=False)
    proba = model.predict(seq, verbose=1)
    y_true = df["label"].astype(int).values
    y_pred = np.argmax(proba, axis=1)

    metrics = save_evaluation_outputs(
        name=name,
        y_true=y_true,
        y_pred=y_pred,
        proba=proba,
        out_dir=out_dir,
    )
    return metrics

import matplotlib.pyplot as plt

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def plot_class_distribution(train_df, val_df, test_df, out_dir: Path):
    ensure_dir(out_dir)

    rows = []
    for split_name, split_df in [
        ("train", train_df),
        ("validation", val_df),
        ("test", test_df),
    ]:
        counts = split_df["label"].value_counts().sort_index()
        for cls in range(NUM_CLASSES):
            rows.append(
                {
                    "split": split_name,
                    "grade": cls,
                    "count": int(counts.get(cls, 0)),
                }
            )

    dist_df = pd.DataFrame(rows)
    dist_df.to_csv(out_dir / "class_distribution_split.csv", index=False)

    pivot = dist_df.pivot(index="grade", columns="split", values="count").fillna(0)

    ax = pivot.plot(kind="bar", figsize=(9, 5))
    ax.set_title("Distribusi Kelas Setelah Split Ulang")
    ax.set_xlabel("Grade KOA")
    ax.set_ylabel("Jumlah Citra")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_dir / "class_distribution_split.png", dpi=200)
    plt.close()

def plot_confusion_matrix_png(cm: np.ndarray, name: str, out_dir: Path):
    ensure_dir(out_dir)

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, interpolation="nearest")

    ax.set_title(f"Confusion Matrix - {name}")
    ax.set_xlabel("Predicted Grade")
    ax.set_ylabel("Actual Grade")
    ax.set_xticks(np.arange(NUM_CLASSES))
    ax.set_yticks(np.arange(NUM_CLASSES))
    ax.set_xticklabels([str(i) for i in range(NUM_CLASSES)])
    ax.set_yticklabels([str(i) for i in range(NUM_CLASSES)])

    thresh = cm.max() / 2.0 if cm.max() > 0 else 0
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(
                j,
                i,
                format(cm[i, j], "d"),
                ha="center",
                va="center",
                color="white" if cm[i, j] > thresh else "black",
            )

    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(out_dir / f"{name}_confusion_matrix.png", dpi=200)
    plt.close()

def save_training_history_outputs(history_obj, phase_name: str, out_dir: Path):
    ensure_dir(out_dir)

    hist_df = pd.DataFrame(history_obj.history)
    hist_df.index.name = "epoch"
    hist_df.to_csv(out_dir / f"{phase_name}_training_history.csv")

    if "loss" in hist_df.columns:
        plt.figure(figsize=(8, 5))
        plt.plot(hist_df.index + 1, hist_df["loss"], label="train_loss")
        if "val_loss" in hist_df.columns:
            plt.plot(hist_df.index + 1, hist_df["val_loss"], label="val_loss")
        plt.title(f"Training Loss - {phase_name}")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(out_dir / f"{phase_name}_loss_curve.png", dpi=200)
        plt.close()

    if "accuracy" in hist_df.columns:
        plt.figure(figsize=(8, 5))
        plt.plot(hist_df.index + 1, hist_df["accuracy"], label="train_accuracy")
        if "val_accuracy" in hist_df.columns:
            plt.plot(hist_df.index + 1, hist_df["val_accuracy"], label="val_accuracy")
        plt.title(f"Training Accuracy - {phase_name}")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(out_dir / f"{phase_name}_accuracy_curve.png", dpi=200)
        plt.close()

    if "auc" in hist_df.columns:
        plt.figure(figsize=(8, 5))
        plt.plot(hist_df.index + 1, hist_df["auc"], label="train_auc")
        if "val_auc" in hist_df.columns:
            plt.plot(hist_df.index + 1, hist_df["val_auc"], label="val_auc")
        plt.title(f"Training AUC - {phase_name}")
        plt.xlabel("Epoch")
        plt.ylabel("AUC")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(out_dir / f"{phase_name}_auc_curve.png", dpi=200)
        plt.close()

def plot_ga_outputs(models_dir: Path, out_dir: Path):
    ensure_dir(out_dir)

    ga_log_path = models_dir / "ga_feature_selection_log.csv"
    selected_path = models_dir / "selected_features_ga_indices.csv"

    if ga_log_path.exists():
        ga_df = pd.read_csv(ga_log_path)

        plt.figure(figsize=(8, 5))
        plt.plot(ga_df["generation"], ga_df["best_score"], label="best_score")
        plt.plot(ga_df["generation"], ga_df["mean_score"], label="mean_score")
        plt.title("Perkembangan Fitness Genetic Algorithm")
        plt.xlabel("Generation")
        plt.ylabel("Fitness Score")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(out_dir / "ga_fitness_curve.png", dpi=200)
        plt.close()

        plt.figure(figsize=(8, 5))
        plt.plot(ga_df["generation"], ga_df["global_best_n_features"], label="global_best_n_features")
        plt.title("Jumlah Fitur Terpilih Selama GA")
        plt.xlabel("Generation")
        plt.ylabel("Jumlah Fitur")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(out_dir / "ga_selected_feature_count_curve.png", dpi=200)
        plt.close()

    if selected_path.exists():
        sel_df = pd.read_csv(selected_path)

        plt.figure(figsize=(10, 3.8))
        plt.scatter(
            sel_df["selected_feature_index"],
            np.ones(len(sel_df)),
            marker="|",
            s=160,
        )
        plt.title("Indeks Fitur VGG19 yang Dipilih oleh GA")
        plt.xlabel("Original VGG19 Bottleneck Feature Index")
        plt.yticks([])
        plt.grid(axis="x", alpha=0.3)
        plt.tight_layout()
        plt.savefig(out_dir / "selected_features_ga_indices.png", dpi=200)
        plt.close()

def save_feature_csvs(
    split_name: str,
    split_df: pd.DataFrame,
    raw_features: np.ndarray,
    scaled_features: np.ndarray,
    selected_indices: np.ndarray,
    out_dir: Path,
):
    ensure_dir(out_dir)

    meta_cols = ["filepath", "filename", "group_id", "label"]
    meta = split_df[meta_cols].reset_index(drop=True).copy()

    raw_cols = [f"vgg19_raw_f{i:03d}" for i in range(raw_features.shape[1])]
    scaled_cols = [f"vgg19_scaled_f{i:03d}" for i in range(scaled_features.shape[1])]

    raw_df = pd.concat([meta, pd.DataFrame(raw_features, columns=raw_cols)], axis=1)
    raw_df.to_csv(out_dir / f"{split_name}_vgg19_raw_features.csv", index=False)

    scaled_df = pd.concat([meta, pd.DataFrame(scaled_features, columns=scaled_cols)], axis=1)
    scaled_df.to_csv(out_dir / f"{split_name}_vgg19_scaled_features.csv", index=False)

    selected_indices = np.asarray(selected_indices).astype(int)

    selected_raw_cols = [f"selected_raw_f{idx:03d}" for idx in selected_indices]
    selected_scaled_cols = [f"selected_scaled_f{idx:03d}" for idx in selected_indices]

    selected_raw_df = pd.concat(
        [meta, pd.DataFrame(raw_features[:, selected_indices], columns=selected_raw_cols)],
        axis=1,
    )
    selected_raw_df.to_csv(out_dir / f"{split_name}_ga_selected_raw_features.csv", index=False)

    selected_scaled_df = pd.concat(
        [meta, pd.DataFrame(scaled_features[:, selected_indices], columns=selected_scaled_cols)],
        axis=1,
    )
    selected_scaled_df.to_csv(out_dir / f"{split_name}_ga_selected_scaled_features.csv", index=False)

def save_preprocessing_preview_images(df: pd.DataFrame, out_dir: Path, n_per_class: int = 3):
    ensure_dir(out_dir)

    sample_rows = []
    for cls in range(NUM_CLASSES):
        cls_df = df[df["label"] == cls]
        if len(cls_df) == 0:
            continue
        sample_rows.append(cls_df.sample(n=min(n_per_class, len(cls_df)), random_state=42))

    if not sample_rows:
        return

    sample_df = pd.concat(sample_rows, ignore_index=True)

    processed_images = []
    titles = []

    for _, row in sample_df.iterrows():
        image_bgr = cv2.imread(row["filepath"], cv2.IMREAD_COLOR)
        if image_bgr is None:
            continue
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        resized = resize_force_rgb(image_rgb, IMG_SIZE)
        clahe_img = apply_clahe_rgb(resized)

        processed_images.append(clahe_img)
        titles.append(f"G{int(row['label'])}\\n{row['filename']}")

        individual_name = f"preview_grade_{int(row['label'])}_{Path(row['filename']).stem}.png"
        cv2.imwrite(str(out_dir / individual_name), cv2.cvtColor(clahe_img, cv2.COLOR_RGB2BGR))

    if not processed_images:
        return

    n = len(processed_images)
    cols = min(5, n)
    rows = int(np.ceil(n / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for i, img in enumerate(processed_images):
        axes[i].imshow(img, cmap="gray")
        axes[i].set_title(titles[i], fontsize=8)
        axes[i].axis("off")

    plt.tight_layout()
    plt.savefig(out_dir / "preprocessing_preview_grid.png", dpi=200)
    plt.close()

def _safe_close_plot():
    try:
        plt.close()
    except Exception:
        pass

def plot_overall_dataset_distribution(all_df: pd.DataFrame, out_dir: Path):
    ensure_dir(out_dir)

    rows = []
    image_counts = all_df["label"].value_counts().sort_index()
    for cls in range(NUM_CLASSES):
        rows.append({"grade": cls, "count": int(image_counts.get(cls, 0))})
    overall_df = pd.DataFrame(rows)
    overall_df.to_csv(out_dir / "01_dataset_scan_class_distribution.csv", index=False)

    plt.figure(figsize=(8, 5))
    plt.bar(overall_df["grade"].astype(str), overall_df["count"])
    plt.title("Distribusi Kelas Dataset Awal")
    plt.xlabel("Grade KOA")
    plt.ylabel("Jumlah Citra")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_dir / "01_dataset_scan_class_distribution.png", dpi=200)
    _safe_close_plot()

    if "group_id" in all_df.columns:
        group_df = all_df.groupby("label")["group_id"].nunique().reindex(range(NUM_CLASSES), fill_value=0)
        group_count_df = pd.DataFrame(
            [{"grade": cls, "unique_group_id": int(group_df.loc[cls])} for cls in range(NUM_CLASSES)]
        )
        group_count_df.to_csv(out_dir / "01_dataset_scan_group_distribution.csv", index=False)

        plt.figure(figsize=(8, 5))
        plt.bar(group_count_df["grade"].astype(str), group_count_df["unique_group_id"])
        plt.title("Distribusi Group ID per Kelas Dataset Awal")
        plt.xlabel("Grade KOA")
        plt.ylabel("Jumlah Unique Group ID")
        plt.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.savefig(out_dir / "01_dataset_scan_group_distribution.png", dpi=200)
        _safe_close_plot()

def plot_split_size_summary(split_df: pd.DataFrame, out_dir: Path):
    ensure_dir(out_dir)

    split_counts = split_df["split"].value_counts().reindex(["train", "val", "test"]).fillna(0).astype(int)
    split_counts.to_csv(out_dir / "02_split_size_summary.csv", header=["count"])

    plt.figure(figsize=(7, 5))
    plt.bar(split_counts.index.astype(str), split_counts.values)
    plt.title("Jumlah Citra per Split")
    plt.xlabel("Split")
    plt.ylabel("Jumlah Citra")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_dir / "02_split_size_summary.png", dpi=200)
    _safe_close_plot()

    pivot = (
        split_df.groupby(["split", "label"])
        .size()
        .unstack(fill_value=0)
        .reindex(index=["train", "val", "test"], columns=list(range(NUM_CLASSES)), fill_value=0)
    )
    pivot.to_csv(out_dir / "02_split_class_matrix.csv")

    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(pivot.values, aspect="auto")
    ax.set_title("Matriks Distribusi Kelas per Split")
    ax.set_xlabel("Grade KOA")
    ax.set_ylabel("Split")
    ax.set_xticks(np.arange(NUM_CLASSES))
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_xticklabels([str(i) for i in range(NUM_CLASSES)])
    ax.set_yticklabels(pivot.index.tolist())
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, str(int(pivot.values[i, j])), ha="center", va="center")
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(out_dir / "02_split_class_matrix.png", dpi=200)
    _safe_close_plot()

def plot_leakage_audit(leakage_report: Dict[str, int], out_dir: Path):
    ensure_dir(out_dir)

    leakage_df = pd.DataFrame(
        [{"pair": key, "overlap_count": int(value)} for key, value in leakage_report.items()]
    )
    leakage_df.to_csv(out_dir / "03_group_leakage_audit.csv", index=False)

    plt.figure(figsize=(8, 4.5))
    plt.bar(leakage_df["pair"], leakage_df["overlap_count"])
    plt.title("Audit Group Leakage antar Split")
    plt.xlabel("Pasangan Split")
    plt.ylabel("Jumlah Group ID Overlap")
    plt.xticks(rotation=20, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_dir / "03_group_leakage_audit.png", dpi=200)
    _safe_close_plot()

def plot_sample_weight_map(sample_weight_map: Dict[int, float], out_dir: Path):
    ensure_dir(out_dir)

    rows = [{"grade": int(k), "sample_weight": float(v)} for k, v in sorted(sample_weight_map.items())]
    weight_df = pd.DataFrame(rows)
    weight_df.to_csv(out_dir / "04_sample_weight_map.csv", index=False)

    plt.figure(figsize=(8, 5))
    plt.bar(weight_df["grade"].astype(str), weight_df["sample_weight"])
    plt.title("Sample Weight per Kelas Training")
    plt.xlabel("Grade KOA")
    plt.ylabel("Sample Weight")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_dir / "04_sample_weight_map.png", dpi=200)
    _safe_close_plot()

def _deprocess_vgg19_image(x: np.ndarray) -> np.ndarray:
    img = x.copy().astype(np.float32)
    img[..., 0] += 103.939
    img[..., 1] += 116.779
    img[..., 2] += 123.68
    img = img[..., ::-1]
    return np.clip(img, 0, 255).astype(np.uint8)

def plot_sequence_batch_preview(seq, out_dir: Path, file_name: str = "05_generator_batch_preview.png", max_images: int = 8):
    ensure_dir(out_dir)

    try:
        batch = seq[0]
        if len(batch) == 3:
            x_batch, y_batch, _ = batch
        else:
            x_batch, y_batch = batch
    except Exception as exc:
        print(f"[WARNING] Gagal membuat batch preview generator: {exc}")
        return

    n = min(max_images, len(x_batch))
    if n <= 0:
        return

    cols = min(4, n)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for i in range(n):
        img = _deprocess_vgg19_image(x_batch[i])
        label = int(np.argmax(y_batch[i]))
        axes[i].imshow(img)
        axes[i].set_title(f"Grade {label}", fontsize=9)
        axes[i].axis("off")

    plt.tight_layout()
    plt.savefig(out_dir / file_name, dpi=200)
    _safe_close_plot()

def plot_feature_extraction_summary(
    x_train_raw: np.ndarray,
    x_val_raw: np.ndarray,
    x_test_raw: np.ndarray,
    x_train_scaled: np.ndarray,
    x_val_scaled: np.ndarray,
    x_test_scaled: np.ndarray,
    out_dir: Path,
):
    ensure_dir(out_dir)

    feature_summary = pd.DataFrame(
        [
            {"split": "train", "n_samples": x_train_raw.shape[0], "n_features_raw": x_train_raw.shape[1], "n_features_scaled": x_train_scaled.shape[1]},
            {"split": "validation", "n_samples": x_val_raw.shape[0], "n_features_raw": x_val_raw.shape[1], "n_features_scaled": x_val_scaled.shape[1]},
            {"split": "test", "n_samples": x_test_raw.shape[0], "n_features_raw": x_test_raw.shape[1], "n_features_scaled": x_test_scaled.shape[1]},
        ]
    )
    feature_summary.to_csv(out_dir / "06_feature_extraction_shape_summary.csv", index=False)

    plt.figure(figsize=(7, 5))
    plt.bar(feature_summary["split"], feature_summary["n_samples"])
    plt.title("Jumlah Sampel Fitur VGG19 per Split")
    plt.xlabel("Split")
    plt.ylabel("Jumlah Sampel")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_dir / "06_feature_extraction_sample_count.png", dpi=200)
    _safe_close_plot()

    raw_mean = np.mean(x_train_raw, axis=0)
    scaled_mean = np.mean(x_train_scaled, axis=0)
    raw_std = np.std(x_train_raw, axis=0)
    scaled_std = np.std(x_train_scaled, axis=0)

    stats_df = pd.DataFrame(
        {
            "feature_index": np.arange(x_train_raw.shape[1]),
            "raw_mean": raw_mean,
            "raw_std": raw_std,
            "scaled_mean": scaled_mean,
            "scaled_std": scaled_std,
        }
    )
    stats_df.to_csv(out_dir / "06_feature_statistics_train.csv", index=False)

    plt.figure(figsize=(9, 5))
    plt.plot(stats_df["feature_index"], stats_df["raw_mean"], label="raw_mean")
    plt.plot(stats_df["feature_index"], stats_df["scaled_mean"], label="scaled_mean")
    plt.title("Rata-rata Fitur Train Sebelum dan Sesudah Scaling")
    plt.xlabel("Index Fitur VGG19")
    plt.ylabel("Mean")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / "06_feature_mean_before_after_scaling.png", dpi=200)
    _safe_close_plot()

    plt.figure(figsize=(9, 5))
    plt.plot(stats_df["feature_index"], stats_df["raw_std"], label="raw_std")
    plt.plot(stats_df["feature_index"], stats_df["scaled_std"], label="scaled_std")
    plt.title("Standar Deviasi Fitur Train Sebelum dan Sesudah Scaling")
    plt.xlabel("Index Fitur VGG19")
    plt.ylabel("Standard Deviation")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / "06_feature_std_before_after_scaling.png", dpi=200)
    _safe_close_plot()

def plot_selected_feature_mask(selected_indices: np.ndarray, total_features: int, out_dir: Path):
    ensure_dir(out_dir)

    selected_indices = np.asarray(selected_indices).astype(int)
    mask = np.zeros(total_features, dtype=int)
    mask[selected_indices] = 1

    mask_df = pd.DataFrame({"feature_index": np.arange(total_features), "selected_by_ga": mask})
    mask_df.to_csv(out_dir / "07_ga_selected_feature_mask.csv", index=False)

    plt.figure(figsize=(10, 3.8))
    plt.bar(mask_df["feature_index"], mask_df["selected_by_ga"])
    plt.title(f"Mask Fitur Terpilih GA ({len(selected_indices)} dari {total_features} fitur)")
    plt.xlabel("Index Fitur VGG19")
    plt.ylabel("Terpilih")
    plt.yticks([0, 1])
    plt.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    plt.savefig(out_dir / "07_ga_selected_feature_mask.png", dpi=200)
    _safe_close_plot()

def plot_rf_feature_importance(rf_model, selected_indices: np.ndarray, out_dir: Path, top_n: int = 25):
    ensure_dir(out_dir)

    if not hasattr(rf_model, "feature_importances_"):
        return

    importances = np.asarray(rf_model.feature_importances_)
    selected_indices = np.asarray(selected_indices).astype(int)

    rows = []
    for local_idx, importance in enumerate(importances):
        original_idx = int(selected_indices[local_idx]) if local_idx < len(selected_indices) else int(local_idx)
        rows.append(
            {
                "local_feature_index": int(local_idx),
                "original_vgg19_feature_index": original_idx,
                "importance": float(importance),
            }
        )

    importance_df = pd.DataFrame(rows).sort_values("importance", ascending=False)
    importance_df.to_csv(out_dir / "08_rf_ga_feature_importance.csv", index=False)

    top_df = importance_df.head(top_n).iloc[::-1]
    plt.figure(figsize=(8, max(5, 0.28 * len(top_df))))
    plt.barh(top_df["original_vgg19_feature_index"].astype(str), top_df["importance"])
    plt.title(f"Top {len(top_df)} Feature Importance RF-GA")
    plt.xlabel("Importance")
    plt.ylabel("Original VGG19 Feature Index")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_dir / "08_rf_ga_top_feature_importance.png", dpi=200)
    _safe_close_plot()

def plot_model_metric_comparison(metrics_summary: Dict[str, Dict[str, Optional[float]]], out_dir: Path):
    ensure_dir(out_dir)

    rows = []
    for model_split, metrics in metrics_summary.items():
        row = {"model_split": model_split}
        row.update(metrics)
        rows.append(row)

    if not rows:
        return

    metrics_df = pd.DataFrame(rows)
    metrics_df.to_csv(out_dir / "09_model_metric_comparison.csv", index=False)

    metrics_to_plot = [
        "accuracy",
        "macro_f1",
        "weighted_f1",
        "quadratic_weighted_kappa",
        "mae_grade",
        "mse_grade",
    ]

    for metric in metrics_to_plot:
        if metric not in metrics_df.columns:
            continue

        plot_df = metrics_df[["model_split", metric]].dropna().copy()
        if plot_df.empty:
            continue

        plt.figure(figsize=(12, 5))
        plt.bar(plot_df["model_split"], plot_df[metric])
        plt.title(f"Perbandingan Model - {metric}")
        plt.xlabel("Model dan Split")
        plt.ylabel(metric)
        plt.xticks(rotation=45, ha="right")
        plt.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        safe_metric = metric.replace("/", "_")
        plt.savefig(out_dir / f"09_model_metric_comparison_{safe_metric}.png", dpi=200)
        _safe_close_plot()

def plot_runtime_log_summary(out_dir: Path):
    visual_dir = ensure_dir(out_dir / "visual_outputs")
    candidates = [
        out_dir / "runtime_log.csv",
        out_dir / "runtime" / "runtime_log.csv",
    ]
    runtime_path = next((p for p in candidates if p.exists()), None)

    if runtime_path is None:
        return

    runtime_df = pd.read_csv(runtime_path)
    if runtime_df.empty or "process" not in runtime_df.columns or "elapsed_seconds" not in runtime_df.columns:
        return

    runtime_df.to_csv(visual_dir / "10_runtime_log_summary.csv", index=False)

    plt.figure(figsize=(12, max(5, 0.35 * len(runtime_df))))
    plt.barh(runtime_df["process"], runtime_df["elapsed_seconds"])
    plt.title("Runtime Setiap Proses")
    plt.xlabel("Elapsed Seconds")
    plt.ylabel("Process")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(visual_dir / "10_runtime_log_summary.png", dpi=200)
    _safe_close_plot()

def plot_standard_scaler_diagnostics(
    x_train_raw,
    x_train_scaled,
    out_dir,
    feature_indices=(0, 50, 100),
):
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    x_train_raw = np.asarray(x_train_raw)
    x_train_scaled = np.asarray(x_train_scaled)

    if x_train_raw.ndim != 2 or x_train_scaled.ndim != 2:
        raise ValueError("x_train_raw dan x_train_scaled harus berupa array 2D.")

    if x_train_raw.shape != x_train_scaled.shape:
        raise ValueError(
            f"Shape x_train_raw {x_train_raw.shape} berbeda dengan "
            f"x_train_scaled {x_train_scaled.shape}."
        )

    n_features = x_train_raw.shape[1]
    valid_indices = [idx for idx in feature_indices if 0 <= idx < n_features]

    if len(valid_indices) == 0:
        valid_indices = sorted(set([0, n_features // 2, n_features - 1]))

    raw_mean = np.mean(x_train_raw, axis=0)
    raw_std = np.std(x_train_raw, axis=0)
    raw_min = np.min(x_train_raw, axis=0)
    raw_max = np.max(x_train_raw, axis=0)
    raw_skew = pd.DataFrame(x_train_raw).skew(axis=0).to_numpy()

    scaled_mean = np.mean(x_train_scaled, axis=0)
    scaled_std = np.std(x_train_scaled, axis=0)
    scaled_min = np.min(x_train_scaled, axis=0)
    scaled_max = np.max(x_train_scaled, axis=0)
    scaled_skew = pd.DataFrame(x_train_scaled).skew(axis=0).to_numpy()

    diagnostics_df = pd.DataFrame(
        {
            "feature_index": np.arange(n_features),
            "raw_mean": raw_mean,
            "raw_std": raw_std,
            "raw_min": raw_min,
            "raw_max": raw_max,
            "raw_skew": raw_skew,
            "scaled_mean": scaled_mean,
            "scaled_std": scaled_std,
            "scaled_min": scaled_min,
            "scaled_max": scaled_max,
            "scaled_skew": scaled_skew,
            "abs_scaled_mean": np.abs(scaled_mean),
            "abs_scaled_std_minus_1": np.abs(scaled_std - 1.0),
        }
    )

    diagnostics_path = out_dir / "06b_standard_scaler_diagnostics_train.csv"
    diagnostics_df.to_csv(diagnostics_path, index=False)

    n_cols = len(valid_indices)
    fig, axes = plt.subplots(
        2,
        n_cols,
        figsize=(5 * n_cols, 7),
        squeeze=False,
    )

    fig.suptitle(
        "Distribusi Fitur Sebelum dan Sesudah StandardScaler\n"
        "(berdasarkan data training)",
        fontsize=13,
        fontweight="bold",
    )

    for col_idx, feature_idx in enumerate(valid_indices):
        raw_values = x_train_raw[:, feature_idx]
        scaled_values = x_train_scaled[:, feature_idx]

        axes[0, col_idx].hist(raw_values, bins=40, alpha=0.85, edgecolor="black")
        axes[0, col_idx].set_title(f"Fitur {feature_idx} — Sebelum Scaling")
        axes[0, col_idx].set_xlabel("Nilai fitur")
        axes[0, col_idx].set_ylabel("Frekuensi")
        axes[0, col_idx].grid(axis="y", alpha=0.25)
        axes[0, col_idx].text(
            0.98,
            0.95,
            f"Mean={raw_mean[feature_idx]:.3f}\n"
            f"Std={raw_std[feature_idx]:.3f}\n"
            f"Skew={raw_skew[feature_idx]:.3f}",
            transform=axes[0, col_idx].transAxes,
            ha="right",
            va="top",
            fontsize=8,
        )

        axes[1, col_idx].hist(scaled_values, bins=40, alpha=0.85, edgecolor="black")
        axes[1, col_idx].set_title(f"Fitur {feature_idx} — Sesudah StandardScaler")
        axes[1, col_idx].set_xlabel("Nilai fitur hasil scaling")
        axes[1, col_idx].set_ylabel("Frekuensi")
        axes[1, col_idx].grid(axis="y", alpha=0.25)
        axes[1, col_idx].text(
            0.98,
            0.95,
            f"Mean={scaled_mean[feature_idx]:.3f}\n"
            f"Std={scaled_std[feature_idx]:.3f}\n"
            f"Skew={scaled_skew[feature_idx]:.3f}",
            transform=axes[1, col_idx].transAxes,
            ha="right",
            va="top",
            fontsize=8,
        )

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    histogram_path = out_dir / "06b_standard_scaler_histogram_before_after.png"
    plt.savefig(histogram_path, dpi=220, bbox_inches="tight")
    plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].boxplot(
        [raw_mean, scaled_mean],
        labels=["Sebelum Scaling", "Sesudah Scaling"],
        showfliers=True,
    )
    axes[0].axhline(0, linestyle="--", linewidth=1)
    axes[0].set_title("Sebaran Mean Fitur")
    axes[0].set_ylabel("Mean")
    axes[0].grid(axis="y", alpha=0.3)

    axes[1].boxplot(
        [raw_std, scaled_std],
        labels=["Sebelum Scaling", "Sesudah Scaling"],
        showfliers=True,
    )
    axes[1].axhline(1, linestyle="--", linewidth=1)
    axes[1].set_title("Sebaran Standard Deviation Fitur")
    axes[1].set_ylabel("Standard Deviation")
    axes[1].grid(axis="y", alpha=0.3)

    plt.suptitle("Ringkasan Efek StandardScaler pada 200 Fitur VGG19", fontweight="bold")
    plt.tight_layout()

    boxplot_path = out_dir / "06b_standard_scaler_mean_std_boxplot.png"
    plt.savefig(boxplot_path, dpi=220, bbox_inches="tight")
    plt.close(fig)

    print("Visualisasi StandardScaler berhasil disimpan:")
    print(f"- {histogram_path}")
    print(f"- {boxplot_path}")
    print(f"- {diagnostics_path}")

    print("\nRingkasan validasi StandardScaler pada train set:")
    print(f"Rata-rata |mean setelah scaling|    : {np.mean(np.abs(scaled_mean)):.8f}")
    print(f"Rata-rata |std setelah scaling - 1| : {np.mean(np.abs(scaled_std - 1.0)):.8f}")

    return diagnostics_df

## 6 — Ekstraksi Fitur VGG19

Fitur diambil dari layer `bottleneck_features`, kemudian disimpan untuk training RF/SVM dan seleksi GA.

In [ ]:
def make_feature_extractor(model: keras.Model) -> keras.Model:
    return keras.Model(
        inputs=model.input,
        outputs=model.get_layer("bottleneck_features").output,
        name="vgg19_feature_extractor",
    )

def extract_features(
    extractor: keras.Model,
    df: pd.DataFrame,
    batch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    seq = XraySequence(df, batch_size=batch_size, augment=False, shuffle=False)
    features = extractor.predict(seq, verbose=1)
    features = np.asarray(features)

    if features.ndim > 2:
        features = features.reshape(features.shape[0], -1)

    labels = df["label"].astype(int).values
    return features.astype(np.float32), labels

## 7 — Komponen Genetic Algorithm Feature Selection

GA hanya menggunakan data training; validation/test tidak ikut dalam proses pemilihan fitur.

In [ ]:
def run_ga_feature_selection(
    x_train_scaled: np.ndarray,
    y_train: np.ndarray,
    seed: int,
    out_dir: Path,
    population_size: int = 28,
    generations: int = 25,
    min_features: int = 10,
    init_prob: float = 0.45,
    mutation_prob: Optional[float] = None,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n_features = x_train_scaled.shape[1]

    if mutation_prob is None:
        mutation_prob = 1.0 / n_features

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=0.25,
        random_state=seed,
    )
    ga_train_idx, ga_val_idx = next(splitter.split(x_train_scaled, y_train))

    x_ga_train = x_train_scaled[ga_train_idx]
    y_ga_train = y_train[ga_train_idx]
    x_ga_val = x_train_scaled[ga_val_idx]
    y_ga_val = y_train[ga_val_idx]

    cache: Dict[bytes, float] = {}

    def repair(mask: np.ndarray) -> np.ndarray:
        mask = mask.astype(bool)
        if mask.sum() < min_features:
            idx = rng.choice(n_features, size=min_features, replace=False)
            mask[:] = False
            mask[idx] = True
        return mask

    def fitness(mask: np.ndarray) -> float:
        mask = repair(mask.copy())
        key = np.packbits(mask.astype(np.uint8)).tobytes()
        if key in cache:
            return cache[key]

        selected = np.where(mask)[0]
        if len(selected) < min_features:
            return 0.0

        clf = ExtraTreesClassifier(
            n_estimators=160,
            max_depth=8,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1,
        )

        try:
            clf.fit(x_ga_train[:, selected], y_ga_train)
            proba = clf.predict_proba(x_ga_val[:, selected])

            full_proba = np.zeros((len(y_ga_val), NUM_CLASSES), dtype=np.float32)
            for col_idx, cls in enumerate(clf.classes_):
                full_proba[:, int(cls)] = proba[:, col_idx]

            auc = roc_auc_score(y_ga_val, full_proba, multi_class="ovr", average="macro")
            size_penalty = 0.015 * (len(selected) / n_features)
            score = float(auc - size_penalty)
        except Exception:
            score = 0.0

        cache[key] = score
        return score

    def tournament_select(population: List[np.ndarray], scores: List[float], k: int = 3) -> np.ndarray:
        idx = rng.choice(len(population), size=k, replace=False)
        best_idx = idx[np.argmax([scores[i] for i in idx])]
        return population[int(best_idx)].copy()

    def crossover(a: np.ndarray, b: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        mask = rng.random(n_features) < 0.5
        c1 = np.where(mask, a, b)
        c2 = np.where(mask, b, a)
        return c1, c2

    def mutate(mask: np.ndarray) -> np.ndarray:
        flips = rng.random(n_features) < mutation_prob
        mask = mask.copy()
        mask[flips] = ~mask[flips]
        return repair(mask)

    population = []
    for _ in range(population_size):
        m = rng.random(n_features) < init_prob
        population.append(repair(m))

    log_rows = []
    best_mask = None
    best_score = -np.inf

    for gen in range(generations):
        scores = [fitness(m) for m in population]
        order = np.argsort(scores)[::-1]

        if scores[int(order[0])] > best_score:
            best_score = float(scores[int(order[0])])
            best_mask = population[int(order[0])].copy()

        log_rows.append(
            {
                "generation": gen + 1,
                "best_score": float(scores[int(order[0])]),
                "mean_score": float(np.mean(scores)),
                "best_n_features": int(population[int(order[0])].sum()),
                "global_best_score": float(best_score),
                "global_best_n_features": int(best_mask.sum()),
            }
        )

        print(
            f"GA Gen {gen + 1:02d}/{generations} | "
            f"best={scores[int(order[0])]:.4f} | "
            f"mean={np.mean(scores):.4f} | "
            f"n_feat={int(population[int(order[0])].sum())}"
        )

        new_population = [population[int(order[0])].copy(), population[int(order[1])].copy()]

        while len(new_population) < population_size:
            p1 = tournament_select(population, scores)
            p2 = tournament_select(population, scores)
            c1, c2 = crossover(p1, p2)
            c1 = mutate(c1)
            c2 = mutate(c2)
            new_population.append(c1)
            if len(new_population) < population_size:
                new_population.append(c2)

        population = new_population

    if best_mask is None:
        raise RuntimeError("GA gagal menemukan subset fitur.")

    pd.DataFrame(log_rows).to_csv(out_dir / "ga_feature_selection_log.csv", index=False)

    selected_indices = np.where(best_mask)[0]
    np.save(out_dir / "selected_features_ga.npy", selected_indices)
    np.save(out_dir / "selected_features_ga_mask.npy", best_mask.astype(bool))

    pd.DataFrame({"selected_feature_index": selected_indices}).to_csv(
        out_dir / "selected_features_ga_indices.csv",
        index=False,
    )

    print(f"GA selesai. Jumlah fitur terpilih: {len(selected_indices)} dari {n_features}")
    return selected_indices

## 8 — Hyperparameter Tuning RF dan SVM

Berisi Random Forest, SVM, dan evaluator sklearn. SVM baseline 200 fitur dilatih pada `x_train_scaled`.

In [ ]:
def make_full_proba(clf, x: np.ndarray) -> np.ndarray:
    proba = clf.predict_proba(x)
    full_proba = np.zeros((x.shape[0], NUM_CLASSES), dtype=np.float32)

    for col_idx, cls in enumerate(clf.classes_):
        full_proba[:, int(cls)] = proba[:, col_idx]

    return full_proba

def tune_random_forest(x_train: np.ndarray, y_train: np.ndarray, seed: int) -> RandomForestClassifier:
    base = RandomForestClassifier(
        random_state=seed,
        class_weight="balanced_subsample",
        n_jobs=-1,
    )

    rf_param_dist = {
        "n_estimators": [100, 200, 300, 500],
        "max_depth": [3, 5, 7, 10, 15, None],
        "min_samples_split": [2, 5, 10, 20, 50],
        "min_samples_leaf": [1, 2, 4, 8, 12],
        "max_features": ["sqrt", "log2", 0.3, 0.5],
        "bootstrap": [True],
        "class_weight": [None, "balanced", "balanced_subsample"],
        "warm_start": [False, True],
        "n_jobs": [1],
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    search = RandomizedSearchCV(
        estimator=base,
        param_distributions=rf_param_dist,
        n_iter=24,
        scoring="roc_auc_ovr",
        cv=cv,
        random_state=seed,
        n_jobs=-1,
        verbose=1,
        refit=True,
    )
    search.fit(x_train, y_train)
    print("Best RF params:", search.best_params_)
    print("Best RF CV score:", search.best_score_)
    return search.best_estimator_

def tune_svm(x_train: np.ndarray, y_train: np.ndarray, seed: int) -> SVC:
    base = SVC(
        probability=True,
        class_weight="balanced",
        random_state=seed,
    )

    svm_param_dist = {
        "C": [0.1, 1, 10, 50, 70, 100],
        "gamma": ["scale", "auto", 0.001, 0.01],
        "kernel": ["linear", "poly", "rbf", "sigmoid"],
        "class_weight": [None, "balanced"],
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    search = RandomizedSearchCV(
        estimator=base,
        param_distributions=svm_param_dist,
        n_iter=15,
        scoring="roc_auc_ovr",
        cv=cv,
        random_state=seed,
        n_jobs=-1,
        verbose=1,
        refit=True,
    )
    search.fit(x_train, y_train)
    print("Best SVM params:", search.best_params_)
    print("Best SVM CV score:", search.best_score_)
    return search.best_estimator_

def evaluate_sklearn_model(
    clf,
    x: np.ndarray,
    y: np.ndarray,
    name: str,
    out_dir: Path,
) -> Dict[str, Optional[float]]:
    y_pred = clf.predict(x)

    if hasattr(clf, "predict_proba"):
        proba = make_full_proba(clf, x)
    else:
        proba = None

    return save_evaluation_outputs(
        name=name,
        y_true=y,
        y_pred=y_pred,
        proba=proba,
        out_dir=out_dir,
    )

## 9 — Runtime Logging

Setiap proses utama dicatat ke `runtime_log.csv` agar durasi eksperimen dapat dilaporkan.

In [ ]:
RUNTIME_LOG_ROWS = []

def seconds_to_hms(seconds: float) -> str:
    seconds = float(seconds)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:05.2f}"

def record_runtime(process_name: str, start_time: float, out_dir: Path, extra_info: str = "") -> float:
    elapsed = time.time() - start_time

    row = {
        "process": process_name,
        "elapsed_seconds": float(elapsed),
        "elapsed_hms": seconds_to_hms(elapsed),
        "extra_info": extra_info,
    }

    RUNTIME_LOG_ROWS.append(row)

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    runtime_df = pd.DataFrame(RUNTIME_LOG_ROWS)
    runtime_df.to_csv(out_dir / "runtime_log.csv", index=False)

    print(f"[RUNTIME] {process_name}: {seconds_to_hms(elapsed)}")
    return elapsed

class EpochTimerCallback(keras.callbacks.Callback):
    def __init__(self, phase_name: str, out_dir: Path):
        super().__init__()
        self.phase_name = phase_name
        self.out_dir = Path(out_dir)
        self.epoch_rows = []
        self.epoch_start = None
        self.out_dir.mkdir(parents=True, exist_ok=True)

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        elapsed = time.time() - self.epoch_start

        row = {
            "phase": self.phase_name,
            "epoch": int(epoch + 1),
            "elapsed_seconds": float(elapsed),
            "elapsed_hms": seconds_to_hms(elapsed),
        }

        if logs:
            for key, value in logs.items():
                try:
                    row[key] = float(value)
                except Exception:
                    row[key] = value

        self.epoch_rows.append(row)

        out_file = self.out_dir / f"{self.phase_name}_epoch_runtime.csv"
        pd.DataFrame(self.epoch_rows).to_csv(out_file, index=False)

        print(f"[EPOCH RUNTIME] {self.phase_name} epoch {epoch + 1}: {seconds_to_hms(elapsed)}")

## 10 — Diagnostik Split Dataset

Opsional. Jalankan untuk mengecek distribusi image dan `group_id` sebelum training penuh.

In [ ]:
preview_df = scan_dataset(Path(args.data_root).resolve())
preview_gtab = group_label_table(preview_df)

print("Jumlah image per kelas:")
print(preview_df["label"].value_counts().sort_index())

print("\nJumlah group_id per kelas:")
print(preview_gtab["group_label"].value_counts().sort_index())

print("\nContoh group_id:")
display(preview_gtab.head(10))

## 11 — Main Pipeline Training dan Evaluasi

Pipeline utama menjalankan scanning dataset, split/balancing, training VGG19, ekstraksi 200 fitur, scaling, GA, Random Forest, SVM GA, RF 200 fitur, dan SVM 200 fitur.

In [ ]:
def main(args) -> None:
    total_start = time.time()
    set_global_seed(args.seed)

    data_root = Path(args.data_root).resolve()
    out_dir = Path(args.out_dir).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    (out_dir / "models").mkdir(exist_ok=True)
    (out_dir / "metrics").mkdir(exist_ok=True)
    (out_dir / "features").mkdir(exist_ok=True)
    (out_dir / "csv_features").mkdir(exist_ok=True)
    (out_dir / "visual_outputs").mkdir(exist_ok=True)
    (out_dir / "preprocessing_preview").mkdir(exist_ok=True)
    (out_dir / "runtime").mkdir(exist_ok=True)

    global RUNTIME_LOG_ROWS
    RUNTIME_LOG_ROWS = []

    print("Scanning dataset...")
    step_start = time.time()
    all_df = scan_dataset(data_root)
    all_df.to_csv(out_dir / "all_images_scanned.csv", index=False)
    record_runtime(
        "01_scan_dataset",
        step_start,
        out_dir / "runtime",
        extra_info=f"n_images={len(all_df)}"
    )

    print(f"Total gambar ditemukan: {len(all_df)}")
    print("Distribusi keseluruhan:")
    print(all_df["label"].value_counts().sort_index())

    plot_overall_dataset_distribution(
        all_df=all_df,
        out_dir=out_dir / "visual_outputs",
    )

    step_start = time.time()
    train_df, val_df, test_df = safe_stratified_group_split(
        all_df,
        train_size=args.train_size,
        val_size=args.val_size,
        test_size=args.test_size,
        seed=args.seed,
    )
    record_runtime(
        "02a_stratified_group_split",
        step_start,
        out_dir / "runtime",
        extra_info=f"train_before_balance={len(train_df)}, val={len(val_df)}, test={len(test_df)}"
    )

    step_start = time.time()
    train_before_balance_counts = train_df["label"].value_counts().sort_index().to_dict()

    train_df = balance_training_dataframe(
        train_df,
        strategy="minority",
        seed=args.seed,
    )

    train_after_balance_counts = train_df["label"].value_counts().sort_index().to_dict()
    record_runtime(
        "02b_balance_training_set",
        step_start,
        out_dir / "runtime",
        extra_info=(
            "strategy=minority; "
            f"before={train_before_balance_counts}; "
            f"after={train_after_balance_counts}"
        )
    )

    step_start = time.time()
    print_distribution("TRAIN", train_df)
    print_distribution("VALIDATION", val_df)
    print_distribution("TEST", test_df)

    split_df = pd.concat(
        [
            train_df.assign(split="train"),
            val_df.assign(split="val"),
            test_df.assign(split="test"),
        ],
        ignore_index=True,
    )
    split_df.to_csv(out_dir / "split_rebuilt_group_aware.csv", index=False)

    leakage_report = {
        "train_val_overlap": len(set(train_df["group_id"]) & set(val_df["group_id"])),
        "train_test_overlap": len(set(train_df["group_id"]) & set(test_df["group_id"])),
        "val_test_overlap": len(set(val_df["group_id"]) & set(test_df["group_id"])),
    }
    with open(out_dir / "split_leakage_audit.json", "w", encoding="utf-8") as f:
        json.dump(leakage_report, f, indent=2)

    print("Leakage audit:", leakage_report)

    plot_split_size_summary(
        split_df=split_df,
        out_dir=out_dir / "visual_outputs",
    )

    plot_leakage_audit(
        leakage_report=leakage_report,
        out_dir=out_dir / "visual_outputs",
    )

    plot_class_distribution(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        out_dir=out_dir / "visual_outputs",
    )

    save_preprocessing_preview_images(
        df=train_df,
        out_dir=out_dir / "preprocessing_preview",
        n_per_class=3,
    )

    record_runtime(
        "02c_save_split_outputs_and_preview",
        step_start,
        out_dir / "runtime",
        extra_info=(
            f"train_after_balance={len(train_df)}, val={len(val_df)}, test={len(test_df)}, "
            f"leakage={leakage_report}"
        )
    )

    step_start = time.time()
    sample_weight_map = compute_sample_weight_map(train_df["label"].astype(int).values)
    print("Sample weight map:", sample_weight_map)

    plot_sample_weight_map(
        sample_weight_map=sample_weight_map,
        out_dir=out_dir / "visual_outputs",
    )

    step_start = time.time()
    augmentation_config = {
        "scenario": "balancing_to_minority_class_with_training_augmentation",
        "applied_to": "training_set_only",
        "validation_augmentation": False,
        "testing_augmentation": False,
        "horizontal_flip": bool(AUGMENTATION_HORIZONTAL_FLIP),
        "rotation_degree": float(AUGMENTATION_ROTATION_DEGREE),
        "zoom_range": float(AUGMENTATION_ZOOM_RANGE),
        "fill_mode": AUGMENTATION_FILL_MODE,
    }

    with open(out_dir / "augmentation_config_minority.json", "w", encoding="utf-8") as f:
        json.dump(augmentation_config, f, indent=2)

    print("Konfigurasi augmentasi training:")
    print(json.dumps(augmentation_config, indent=2))

    record_runtime(
        "03a_configure_training_augmentation",
        step_start,
        out_dir / "runtime",
        extra_info=json.dumps(augmentation_config)
    )

    step_start = time.time()
    train_seq = XraySequence(
        train_df,
        batch_size=args.batch_size,
        augment=True,
        shuffle=True,
        sample_weight_map=sample_weight_map,
    )
    val_seq = XraySequence(
        val_df,
        batch_size=args.batch_size,
        augment=False,
        shuffle=False,
        sample_weight_map=None,
    )
    record_runtime(
        "03_create_data_generators",
        step_start,
        out_dir / "runtime",
        extra_info=f"batch_size={args.batch_size}; train_augment=True; rotation={AUGMENTATION_ROTATION_DEGREE}; zoom={AUGMENTATION_ZOOM_RANGE}; horizontal_flip={AUGMENTATION_HORIZONTAL_FLIP}"
    )

    preview_seq = XraySequence(
        train_df.head(min(args.batch_size, len(train_df))),
        batch_size=max(1, min(args.batch_size, len(train_df))),
        augment=False,
        shuffle=False,
        sample_weight_map=None,
    )
    plot_sequence_batch_preview(
        seq=preview_seq,
        out_dir=out_dir / "visual_outputs",
        file_name="05_train_generator_batch_preview.png",
        max_images=8,
    )

    step_start = time.time()
    augmented_preview_seq = XraySequence(
        train_df.head(min(args.batch_size, len(train_df))),
        batch_size=max(1, min(args.batch_size, len(train_df))),
        augment=True,
        shuffle=False,
        sample_weight_map=None,
    )
    plot_sequence_batch_preview(
        seq=augmented_preview_seq,
        out_dir=out_dir / "visual_outputs",
        file_name="05b_train_generator_augmented_batch_preview.png",
        max_images=8,
    )
    record_runtime(
        "03b_preview_training_augmentation",
        step_start,
        out_dir / "runtime",
        extra_info="preview augmentasi training: horizontal_flip=True, rotation=10_degree, zoom=20_percent"
    )

    print("Building VGG19 model...")
    step_start = time.time()
    model = build_vgg19_model(
        bottleneck_dim=args.bottleneck_dim,
        dropout_rate=args.dropout,
        l2_value=args.l2,
    )
    save_model_summary(model, out_dir / "model_summary_phase1.txt")
    record_runtime(
        "04_build_vgg19_model",
        step_start,
        out_dir / "runtime",
        extra_info=f"bottleneck_dim={args.bottleneck_dim}"
    )

    best_weights_path = out_dir / "models" / "best_vgg19_koa.weights.h5"

    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(best_weights_path),
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True,
            mode="min",
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=args.early_stop_patience,
            restore_best_weights=True,
            mode="min",
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.35,
            patience=4,
            min_lr=1e-7,
            mode="min",
            verbose=1,
        ),
        keras.callbacks.CSVLogger(str(out_dir / "cnn_training_log.csv"), append=False),
    ]

    print("=== Phase 1: train classification head, VGG19 frozen ===")
    compile_model(model, learning_rate=args.lr_head)

    step_start = time.time()
    hist1 = model.fit(
        train_seq,
        validation_data=val_seq,
        epochs=args.epochs_head,
        callbacks=callbacks + [
            EpochTimerCallback(
                phase_name="phase1_head_frozen",
                out_dir=out_dir / "runtime",
            )
        ],
        verbose=1,
    )
    record_runtime(
        "05_training_phase1_head_frozen",
        step_start,
        out_dir / "runtime",
        extra_info=f"epochs_head={args.epochs_head}, lr={args.lr_head}"
    )

    save_training_history_outputs(
        hist1,
        phase_name="phase1_head_frozen",
        out_dir=out_dir / "visual_outputs",
    )

    if args.epochs_finetune > 0:
        print("=== Phase 2: conservative fine-tuning block5 only ===")
        step_start = time.time()
        set_vgg19_trainable_block5_only(model)
        save_model_summary(model, out_dir / "model_summary_phase2.txt")
        compile_model(model, learning_rate=args.lr_finetune)
        record_runtime(
            "06_prepare_phase2_block5_finetune",
            step_start,
            out_dir / "runtime",
            extra_info=f"lr_finetune={args.lr_finetune}"
        )

        step_start = time.time()
        hist2 = model.fit(
            train_seq,
            validation_data=val_seq,
            epochs=args.epochs_finetune,
            callbacks=callbacks + [
                EpochTimerCallback(
                    phase_name="phase2_block5_finetune",
                    out_dir=out_dir / "runtime",
                )
            ],
            verbose=1,
        )
        record_runtime(
            "07_training_phase2_block5_finetune",
            step_start,
            out_dir / "runtime",
            extra_info=f"epochs_finetune={args.epochs_finetune}, lr={args.lr_finetune}"
        )

        save_training_history_outputs(
            hist2,
            phase_name="phase2_block5_finetune",
            out_dir=out_dir / "visual_outputs",
        )

    print("Loading best CNN model before evaluation and feature extraction...")
    step_start = time.time()
    if not best_weights_path.exists():
        raise RuntimeError(f"Best weights tidak ditemukan: {best_weights_path}")

    best_model = build_vgg19_model(
        bottleneck_dim=args.bottleneck_dim,
        dropout_rate=args.dropout,
        l2_value=args.l2,
    )
    best_model.load_weights(best_weights_path)
    print(f"Best weights loaded from: {best_weights_path}")

    best_model.save(out_dir / "models" / "best_vgg19_koa_full.h5", include_optimizer=False)
    record_runtime(
        "08_load_best_weights_and_save_full_model",
        step_start,
        out_dir / "runtime",
        extra_info=str(best_weights_path)
    )

    print("Evaluating CNN classifier...")
    step_start = time.time()
    cnn_train_metrics = evaluate_cnn(best_model, train_df, args.batch_size, "cnn_train", out_dir / "metrics")
    record_runtime("09_evaluate_cnn_train", step_start, out_dir / "runtime", extra_info=str(cnn_train_metrics))

    step_start = time.time()
    cnn_val_metrics = evaluate_cnn(best_model, val_df, args.batch_size, "cnn_val", out_dir / "metrics")
    record_runtime("10_evaluate_cnn_val", step_start, out_dir / "runtime", extra_info=str(cnn_val_metrics))

    step_start = time.time()
    cnn_test_metrics = evaluate_cnn(best_model, test_df, args.batch_size, "cnn_test", out_dir / "metrics")
    record_runtime("11_evaluate_cnn_test", step_start, out_dir / "runtime", extra_info=str(cnn_test_metrics))

    print("CNN Train:", cnn_train_metrics)
    print("CNN Val:", cnn_val_metrics)
    print("CNN Test:", cnn_test_metrics)

    print("Creating and saving feature extractor...")
    step_start = time.time()
    extractor = make_feature_extractor(best_model)

    extractor.save(out_dir / "models" / "vgg19_attention_extractor.h5", include_optimizer=False)
    record_runtime(
        "12_create_and_save_feature_extractor",
        step_start,
        out_dir / "runtime",
        extra_info="vgg19_attention_extractor.h5"
    )

    print("Extracting bottleneck features...")
    step_start = time.time()
    x_train_raw, y_train = extract_features(extractor, train_df, args.batch_size)
    record_runtime("13_extract_train_features", step_start, out_dir / "runtime", extra_info=f"shape={x_train_raw.shape}")

    step_start = time.time()
    x_val_raw, y_val = extract_features(extractor, val_df, args.batch_size)
    record_runtime("14_extract_val_features", step_start, out_dir / "runtime", extra_info=f"shape={x_val_raw.shape}")

    step_start = time.time()
    x_test_raw, y_test = extract_features(extractor, test_df, args.batch_size)
    record_runtime("15_extract_test_features", step_start, out_dir / "runtime", extra_info=f"shape={x_test_raw.shape}")

    step_start = time.time()
    np.save(out_dir / "features" / "x_train_raw_features.npy", x_train_raw)
    np.save(out_dir / "features" / "x_val_raw_features.npy", x_val_raw)
    np.save(out_dir / "features" / "x_test_raw_features.npy", x_test_raw)
    np.save(out_dir / "features" / "y_train.npy", y_train)
    np.save(out_dir / "features" / "y_val.npy", y_val)
    np.save(out_dir / "features" / "y_test.npy", y_test)
    record_runtime("16_save_raw_features_npy", step_start, out_dir / "runtime")

    print("Feature shapes:")
    print("  train:", x_train_raw.shape)
    print("  val:", x_val_raw.shape)
    print("  test:", x_test_raw.shape)

    print("Fitting scaler on TRAIN features only...")
    step_start = time.time()
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train_raw)
    x_val_scaled = scaler.transform(x_val_raw)
    x_test_scaled = scaler.transform(x_test_raw)
    joblib.dump(scaler, out_dir / "models" / "scaler.pkl")

    plot_feature_extraction_summary(
        x_train_raw=x_train_raw,
        x_val_raw=x_val_raw,
        x_test_raw=x_test_raw,
        x_train_scaled=x_train_scaled,
        x_val_scaled=x_val_scaled,
        x_test_scaled=x_test_scaled,
        out_dir=out_dir / "visual_outputs",
    )

    plot_standard_scaler_diagnostics(
        x_train_raw=x_train_raw,
        x_train_scaled=x_train_scaled,
        out_dir=out_dir / "visual_outputs",
        feature_indices=(0, 50, 100),
    )
    record_runtime(
        "17_fit_scaler_and_transform_features",
        step_start,
        out_dir / "runtime",
        extra_info="StandardScaler fit on train only"
    )

    print("Running GA feature selection on TRAIN only...")
    step_start = time.time()
    selected_indices = run_ga_feature_selection(
        x_train_scaled=x_train_scaled,
        y_train=y_train,
        seed=args.seed,
        out_dir=out_dir / "models",
        population_size=args.ga_population,
        generations=args.ga_generations,
        min_features=args.ga_min_features,
        init_prob=args.ga_init_prob,
    )
    record_runtime(
        "18_ga_feature_selection",
        step_start,
        out_dir / "runtime",
        extra_info=f"selected_features={len(selected_indices)}, generations={args.ga_generations}, population={args.ga_population}"
    )

    step_start = time.time()
    plot_ga_outputs(
        models_dir=out_dir / "models",
        out_dir=out_dir / "visual_outputs",
    )

    plot_selected_feature_mask(
        selected_indices=selected_indices,
        total_features=x_train_scaled.shape[1],
        out_dir=out_dir / "visual_outputs",
    )

    save_feature_csvs(
        split_name="train",
        split_df=train_df,
        raw_features=x_train_raw,
        scaled_features=x_train_scaled,
        selected_indices=selected_indices,
        out_dir=out_dir / "csv_features",
    )

    save_feature_csvs(
        split_name="validation",
        split_df=val_df,
        raw_features=x_val_raw,
        scaled_features=x_val_scaled,
        selected_indices=selected_indices,
        out_dir=out_dir / "csv_features",
    )

    save_feature_csvs(
        split_name="test",
        split_df=test_df,
        raw_features=x_test_raw,
        scaled_features=x_test_scaled,
        selected_indices=selected_indices,
        out_dir=out_dir / "csv_features",
    )
    record_runtime(
        "19_save_ga_plots_and_feature_csvs",
        step_start,
        out_dir / "runtime",
        extra_info="visual_outputs and csv_features"
    )

    step_start = time.time()
    x_train_ga = x_train_scaled[:, selected_indices]
    x_val_ga = x_val_scaled[:, selected_indices]
    x_test_ga = x_test_scaled[:, selected_indices]
    record_runtime(
        "20_apply_selected_features_to_splits",
        step_start,
        out_dir / "runtime",
        extra_info=f"selected_feature_count={len(selected_indices)}"
    )

    print("Tuning Random Forest on selected GA features...")
    step_start = time.time()
    rf_ga = tune_random_forest(x_train_ga, y_train, seed=args.seed)
    joblib.dump(rf_ga, out_dir / "models" / "best_rf_ga.pkl")

    plot_rf_feature_importance(
        rf_model=rf_ga,
        selected_indices=selected_indices,
        out_dir=out_dir / "visual_outputs",
        top_n=25,
    )
    record_runtime("21_tune_train_rf_ga", step_start, out_dir / "runtime")

    print("Tuning SVM on selected GA features...")
    step_start = time.time()
    svm_ga = tune_svm(x_train_ga, y_train, seed=args.seed)
    joblib.dump(svm_ga, out_dir / "models" / "best_svm_ga.pkl")
    record_runtime("22_tune_train_svm_ga", step_start, out_dir / "runtime")

    print("Training RF baseline on all 200 features for comparison...")
    step_start = time.time()
    rf_all = tune_random_forest(x_train_scaled, y_train, seed=args.seed)
    joblib.dump(rf_all, out_dir / "models" / "best_rf_all_features.pkl")
    record_runtime("23_tune_train_rf_all_features", step_start, out_dir / "runtime")

    print("Training SVM baseline on all 200 features for comparison...")
    step_start = time.time()
    svm_all = tune_svm(x_train_scaled, y_train, seed=args.seed)
    joblib.dump(svm_all, out_dir / "models" / "best_svm_all_features.pkl")
    record_runtime("24_tune_train_svm_all_200_features", step_start, out_dir / "runtime")

    print("Evaluating final models...")
    step_start = time.time()
    metrics_summary = {}

    for split_name, x_all, x_ga, y in [
        ("train", x_train_scaled, x_train_ga, y_train),
        ("val", x_val_scaled, x_val_ga, y_val),
        ("test", x_test_scaled, x_test_ga, y_test),
    ]:
        metrics_summary[f"rf_ga_{split_name}"] = evaluate_sklearn_model(
            rf_ga,
            x_ga,
            y,
            f"rf_ga_{split_name}",
            out_dir / "metrics",
        )
        metrics_summary[f"svm_ga_{split_name}"] = evaluate_sklearn_model(
            svm_ga,
            x_ga,
            y,
            f"svm_ga_{split_name}",
            out_dir / "metrics",
        )
        metrics_summary[f"rf_all_{split_name}"] = evaluate_sklearn_model(
            rf_all,
            x_all,
            y,
            f"rf_all_{split_name}",
            out_dir / "metrics",
        )
        metrics_summary[f"svm_all_{split_name}"] = evaluate_sklearn_model(
            svm_all,
            x_all,
            y,
            f"svm_all_{split_name}",
            out_dir / "metrics",
        )

    with open(out_dir / "metrics" / "final_metrics_summary.json", "w", encoding="utf-8") as f:
        json.dump(metrics_summary, f, indent=2)

    plot_model_metric_comparison(
        metrics_summary=metrics_summary,
        out_dir=out_dir / "visual_outputs",
    )

    plot_runtime_log_summary(out_dir)

    record_runtime(
        "25_evaluate_final_classifiers",
        step_start,
        out_dir / "runtime",
        extra_info="rf_ga, svm_ga, rf_all, svm_all on train/val/test"
    )

    step_start = time.time()
    with open(out_dir / "models" / "class_names.json", "w", encoding="utf-8") as f:
        json.dump(CLASS_NAMES, f, indent=2)

    metadata = {
        "dataset_root": str(data_root),
        "img_size": IMG_SIZE,
        "preprocessing": "resize 224x224 bicubic -> CLAHE LAB L-channel -> VGG19 preprocess_input",
        "cnn_model": "VGG19 ImageNet frozen + conservative block5 fine-tuning",
        "bottleneck_dim": args.bottleneck_dim,
        "feature_selection": "Genetic Algorithm on training features only",
        "classifier": "Random Forest and SVM with GA-selected features plus all 200 VGG19 features",
        "selected_feature_count": int(len(selected_indices)),
        "all_vgg19_feature_count": int(x_train_scaled.shape[1]),
        "selected_feature_indices": selected_indices.astype(int).tolist(),
        "split_leakage_audit": leakage_report,
    }

    with open(out_dir / "models" / "training_metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    record_runtime("26_save_metadata", step_start, out_dir / "runtime")

    total_elapsed = record_runtime(
        "27_total_pipeline_runtime",
        total_start,
        out_dir / "runtime",
        extra_info="full pipeline"
    )

    print("DONE.")
    print(f"Output folder: {out_dir}")
    print("Model untuk Streamlit:")
    print(f"  {out_dir / 'models' / 'best_rf_ga.pkl'}")
    print(f"  {out_dir / 'models' / 'best_svm_ga.pkl'}")
    print(f"  {out_dir / 'models' / 'best_rf_all_features.pkl'}")
    print(f"  {out_dir / 'models' / 'best_svm_all_features.pkl'}")
    print(f"  {out_dir / 'models' / 'scaler.pkl'}")
    print(f"  {out_dir / 'models' / 'selected_features_ga.npy'}")
    print(f"  {out_dir / 'models' / 'vgg19_attention_extractor.h5'}")
    print(f"Total runtime: {seconds_to_hms(total_elapsed)}")

## 12 — Jalankan Eksperimen

In [ ]:
start = time.time()
main(args)
print(f"Elapsed time: {(time.time() - start) / 60:.2f} minutes")

## 13 — Review Output Akhir

Cell ini dipakai setelah training selesai untuk menampilkan ringkasan metrik, output visual, CSV fitur, dan runtime log.

In [ ]:
models_dir = Path(args.out_dir) / "models"
metrics_dir = Path(args.out_dir) / "metrics"

print("Model files:")
for p in sorted(models_dir.glob("*")):
    print("-", p.name)

print("\nMetric files:")
for p in sorted(metrics_dir.glob("*"))[:30]:
    print("-", p.name)

import json
import pandas as pd

summary_path = Path(args.out_dir) / "metrics" / "final_metrics_summary.json"

if summary_path.exists():
    with open(summary_path, "r", encoding="utf-8") as f:
        metrics_summary = json.load(f)

    rows = []
    for model_name, metrics in metrics_summary.items():
        row = {"model_split": model_name}
        row.update(metrics)
        rows.append(row)

    summary_df = pd.DataFrame(rows)
    display(summary_df)
else:
    print("final_metrics_summary.json belum ada. Jalankan training sampai selesai terlebih dahulu.")

visual_dir = Path(args.out_dir) / "visual_outputs"
preview_dir = Path(args.out_dir) / "preprocessing_preview"
csv_feature_dir = Path(args.out_dir) / "csv_features"

print("Visual outputs (.png):")
for p in sorted(visual_dir.glob("*.png")):
    print("-", p)

print("\nPreprocessing preview images (.png):")
for p in sorted(preview_dir.glob("*.png"))[:30]:
    print("-", p)

print("\nCSV fitur:")
for p in sorted(csv_feature_dir.glob("*.csv")):
    print("-", p)

runtime_dir = Path(args.out_dir) / "runtime"

print("Runtime files:")
for p in sorted(runtime_dir.glob("*.csv")):
    print("-", p)

runtime_log = runtime_dir / "runtime_log.csv"
if runtime_log.exists():
    runtime_df = pd.read_csv(runtime_log)
    display(runtime_df)
else:
    print("runtime_log.csv belum tersedia.")